In [14]:
import os
from getpass import getpass

import pandas as pd
from neo4j import GraphDatabase

# Prefer env vars so you don't hardcode credentials.
# For Neo4j Aura, URI usually looks like: neo4j+s://<id>.databases.neo4j.io
# For local Neo4j Desktop, URI is typically: bolt://localhost:7687
uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
user = os.getenv("NEO4J_USER", "neo4j")
password = os.getenv("NEO4J_PASSWORD") or getpass("Neo4j password: ")

driver = GraphDatabase.driver(uri, auth=(user, password))



In [15]:

def run_cypher(query: str, params: dict | None = None, database: str = "neo4j") -> pd.DataFrame:
    with driver.session(database=database) as session:
        result = session.run(query, params or {})
        return pd.DataFrame([record.data() for record in result])


df = run_cypher("""
MATCH (n)
RETURN labels(n) AS labels, count(*) AS cnt
ORDER BY cnt DESC
LIMIT 10
""")

df

,labels,cnt
0,[Description],1719423
1,[RoleGroup],786366
2,[ObjectConcept],537781


In [16]:
df.count()

labels    3
cnt       3
dtype: int64

## Parameterized Cypher example

Use parameters to avoid string concatenation and keep queries safe/reusable.

In [17]:
# Example: fetch one label safely via parameter
label_name = "Person"
example_df = run_cypher(
    """
    MATCH (n)
    WHERE $label IN labels(n)
    RETURN labels(n) AS labels, count(*) AS cnt
    ORDER BY cnt DESC
    LIMIT 20
    """,
    params={"label": label_name},
)

example_df

""


In [18]:
print(example_df.count())

Series([], dtype: int64)


In [11]:
# Optional: close connection when done
driver.close()